# Data Cleaning of the Flood Control Projects Dataset

In this notebook, we perform data cleaning on the Flood Control Projects dataset that will be used for further analysis and modeling.

The purpose of this notebook is to prepare the dataset so that it is structured, consistent, and suitable for analysis. The cleaning process will include tasks such as:

- Inspecting the dataset structure and identifying potential issues
- Handling missing or incomplete values
- Standardizing column names and data formats
- Converting data types where necessary

After completing these steps, the dataset will be in a form that allows us to perform reliable exploratory data analysis and apply analytical or predictive methods in later stages of the project.

### Import
Start by importing **numpy** and **pandas**.

In [ ]:
# Import relevant python modules
import numpy as np
import pandas as pd 

### Setting Float Display Format

This step configures how floating-point numbers are displayed in the notebook. To improve readability, we modify the pandas display option so that all floating-point numbers are shown in standard decimal format with two decimal places. This makes numerical values easier to interpret when viewing tables and summary statistics in the dataset.

The following line sets the global display format for floating-point values in pandas:
- `{:,.2f}` ensures that numbers are displayed with two decimal places and include commas as thousands separators.

In [ ]:
# Force display options to be in float format instead
# of scientific notations
pd.options.display.float_format = '{:,.2f}'.format

### Loading the Flood Control Projects Dataset

In this step, we load the dataset containing flood control project information into a pandas DataFrame. The dataset is stored in a CSV file, and we use `pd.read_csv()` to read it into memory for analysis.

Once the dataset is loaded, we perform some initial inspections to understand its structure:

- `df_infra.columns` displays the column names, giving an overview of the variables available in the dataset.
- `df_infra.head()` shows the first few rows of the dataset, providing a quick glimpse of the data entries.
- `df_infra.info()` provides a summary of the dataset, including the number of entries, column data types, and the count of non-null values for each column.

These initial checks help us identify potential issues such as missing values, incorrect data types, or unexpected column names before proceeding with further cleaning and analysis.

In [ ]:
# 1. Grab the paths of all CSV files in your folder
path = './data/infra-projects/dpwh_flood_control_projects.csv'
df_infra = pd.read_csv(path)

# Sanity check
df_infra.columns
df_infra.head()
df_infra.info()

### Creating a Clean Subset of Columns

At this stage, we reduce the dataset to only the columns that are relevant for the analysis. The selected columns include:

- **FundingYear** – the year the project received funding  
- **Region** – the administrative region where the project is located  
- **Province** – the province where the project is implemented  
- **Municipality** – the municipality or city where the project is located  
- **ApprovedBudgetForContract** – the approved budget allocated for the project  
- **ContractCost** – the actual contracted cost of the project  
- **ActualCompletionDate** – the recorded date when the project was completed  
- **TypeOfWork** – the category or type of flood control work performed  

By creating this subset and assigning it to a new DataFrame (`df_infra`), we ensure that the dataset contains only the variables needed for the subsequent cleaning and analysis steps. The `.copy()` method is used to explicitly create a separate copy of the selected data, preventing unintended modifications to the original dataset.

In [ ]:
# 2. Create a clean subset
df_infra = df_infra[[
    'FundingYear', 'Region', 'Province', 'Municipality', 
    'ApprovedBudgetForContract', 'ContractCost', 
    'ActualCompletionDate', 'TypeOfWork'
]].copy()

### Converting Budget Columns to Numeric Data Types

In this step, the budget-related columns are converted to numeric data types to ensure they can be properly used for calculations and analysis. In some datasets, numerical values may be stored as strings due to formatting issues, missing values, or inconsistencies in the source data.

The `pd.to_numeric()` function is used to convert the following columns into numeric format:

- **ApprovedBudgetForContract**
- **ContractCost**

The parameter `errors='coerce'` ensures that any values that cannot be properly converted (such as invalid strings or corrupted entries) are automatically replaced with `NaN`. This allows the dataset to remain usable while clearly marking problematic values that may need to be handled later in the data cleaning process.

Ensuring that these columns are stored as numeric values is important for performing accurate operations such as aggregations, comparisons, and statistical analysis.

In [ ]:
# 3. Convert to numeric (handles cases where they might still be strings)
df_infra['ApprovedBudgetForContract'] = pd.to_numeric(df_infra['ApprovedBudgetForContract'], errors='coerce')
df_infra['ContractCost'] = pd.to_numeric(df_infra['ContractCost'], errors='coerce')

### Creating the Final Budget Column

In this step, a new column called **Final_Budget** is created to represent the most reliable available budget value for each project. Some records may have missing values in the **ContractCost** column, which can make it difficult to consistently analyze project costs.

To address this, a fallback logic is applied using the `fillna()` method. The process works as follows:

- If **ContractCost** is available, it is used as the value for **Final_Budget**.
- If **ContractCost** is missing (`NaN`), the value from **ApprovedBudgetForContract** is used instead.

This approach ensures that each record has a usable budget value whenever possible, improving the completeness of the dataset and allowing more consistent financial analysis in later stages of the project.

In [ ]:
# 4. Create your 'Final_Budget' column using the fallback logic
df_infra['Final_Budget'] = df_infra['ContractCost'].fillna(df_infra['ApprovedBudgetForContract'])

### Calculating Budget Variance

In this step, a new column called **Budget_Variance** is created to measure the difference between the approved project budget and the actual contract cost.

The value is calculated by subtracting **ContractCost** from **ApprovedBudgetForContract**:

- **ApprovedBudgetForContract** – the initially approved budget for the project  
- **ContractCost** – the final contracted cost for completing the project  

The resulting **Budget_Variance** indicates how much the project cost differed from the originally approved budget. A positive value suggests that the contract cost was lower than the approved budget (indicating potential savings), while a negative value indicates that the contract cost exceeded the approved budget.

This metric can be useful for analyzing budgeting efficiency and identifying patterns in project cost management across different regions or types of work.

In [ ]:
# 5. Create a 'Savings' column (The difference between ABC and ContractCost)
df_infra['Budget_Variance'] = df_infra['ApprovedBudgetForContract'] - df_infra['ContractCost']

### Converting Budget Values to Millions

In this step, the **Final_Budget** values are converted into millions to make the figures easier to interpret and report in the analysis. Large monetary values can be difficult to read when expressed in full units. By dividing the **Final_Budget** column by **1,000,000**, a new column called **Final_Budget_M** is created, where the values represent the budget in millions of currency units.

In [ ]:
# 6. Convert to Millions for your final study reporting
df_infra['Final_Budget_M'] = df_infra['Final_Budget'] / 1_000_000

### Saving the Cleaned Dataset

After completing the data cleaning and transformation steps, the processed DataFrame is saved as a new CSV file. This allows the cleaned dataset to be reused in later stages of the project without repeating the entire cleaning process.

In [ ]:
# Save your cleaned dataframe to a specific folder
# 'index=False' prevents pandas from adding an extra column of numbers at the start
df_infra.to_csv('data/infra-projects/cleaned_infra_projects.csv', index=False)

